In [1]:
from sys import path

path.append("..")

import os
from pathlib import Path

os.chdir(Path.cwd().parent)

In [2]:
import re
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import psutil

from src.graph_utils.dataset import Dataset
from src.graph_utils.reading import read_graph6
from src.descriptors.edge_descriptors import edge_descriptors_dict
from src.descriptors.node_descriptors import node_descriptors_dict
from src.collision_tests.comparison import TestOperator

from src.settings import Settings


In [5]:
for filename in os.listdir(Settings.raw_datasets_dir):
    if '.txt' in filename:
        old_filename = os.path.join('raw_datasets', filename)
        new_filename = re.sub(r'.txt', '', old_filename)
        print(old_filename, new_filename)

        os.rename(old_filename, new_filename)

In [6]:
psutil.cpu_count()

12

In [7]:
list(edge_descriptors_dict.keys()) + list(node_descriptors_dict.keys())

['jaccard_index',
 'jaccard_index_normalized',
 'adamic_adar',
 'katz_index',
 'neighborhood_distance',
 'neighborhood_measure',
 'preferential_attachment',
 'resource_allocation',
 'same_community',
 'total_nieghbors',
 'algebraic_distance',
 'ari',
 'lds',
 'scan',
 'cn_quadrangle',
 'cn_triangle',
 'lss',
 'simmelian_sparsifier_np',
 'edge_betweenness',
 'edge_betweenness_normalized',
 'spanning_edge',
 'ldp_degree',
 'ldp_min',
 'ldp_min_normalized',
 'ldp_max',
 'ldp_max_normalized',
 'ldp_mean',
 'ldp_mean_normalized',
 'ldp_std',
 'ldp_std_normalized',
 'eigenvector_centrality',
 'closeness',
 'closeness_normalized',
 'degree_centrality',
 'degree_centrality_normalized',
 'katz_centrality',
 'lcc']

In [8]:
single_features = [(k, ) for k in edge_descriptors_dict.keys()]

node_features = [(f, ) for f in list(filter(lambda k: 'ldp' not in k, node_descriptors_dict.keys()))]
node_features.extend([tuple(filter(lambda k: 'ldp' in k and 'normalized' not in k, node_descriptors_dict.keys()))])
node_features.extend([tuple(filter(lambda k: 'ldp' in k and ('normalized' in k or 'degree' in k), node_descriptors_dict.keys()))])

single_features.extend(node_features)

all_features = []

all_features.extend(single_features \
    + list(map(lambda params: tuple(map(lambda x: x + ':k_graph2', params)), single_features)) \
    + list(map(lambda params: tuple(map(lambda x: x + ':k_graph3', params)), single_features)) \
    + list(map(lambda params: tuple(map(lambda x: x + ':modk_graph3', params)), single_features)) \
)

all_features.extend(list(map(lambda features: tuple(x + k_graph for k_graph in ['',':k_graph2',':k_graph3'] for x in features), single_features)))

all_features


[('jaccard_index',),
 ('jaccard_index_normalized',),
 ('adamic_adar',),
 ('katz_index',),
 ('neighborhood_distance',),
 ('neighborhood_measure',),
 ('preferential_attachment',),
 ('resource_allocation',),
 ('same_community',),
 ('total_nieghbors',),
 ('algebraic_distance',),
 ('ari',),
 ('lds',),
 ('scan',),
 ('cn_quadrangle',),
 ('cn_triangle',),
 ('lss',),
 ('simmelian_sparsifier_np',),
 ('edge_betweenness',),
 ('edge_betweenness_normalized',),
 ('spanning_edge',),
 ('eigenvector_centrality',),
 ('closeness',),
 ('closeness_normalized',),
 ('degree_centrality',),
 ('degree_centrality_normalized',),
 ('katz_centrality',),
 ('lcc',),
 ('ldp_degree', 'ldp_min', 'ldp_max', 'ldp_mean', 'ldp_std'),
 ('ldp_degree',
  'ldp_min_normalized',
  'ldp_max_normalized',
  'ldp_mean_normalized',
  'ldp_std_normalized'),
 ('jaccard_index:k_graph2',),
 ('jaccard_index_normalized:k_graph2',),
 ('adamic_adar:k_graph2',),
 ('katz_index:k_graph2',),
 ('neighborhood_distance:k_graph2',),
 ('neighborhood_me

In [9]:
import os
from typing import List
from src.graph_utils.dataset import Dataset
from src.collision_tests.utils import TestParameters

arguments_list = []
datasets: List[Dataset] = []
# for dataset_name in ['graph5', 'graph6', 'graph7']:
# for dataset_name in ['regular']:

for dataset_name in  list(map(lambda x: '.'.join(x.split('.')[:-1]), filter(lambda x: '.g6' in x, os.listdir(Settings.raw_datasets_dir))))[:15]:

    dataset = Dataset(name=dataset_name)
    datasets.append(dataset)
    if len(dataset) > 100_000:
        continue
    for features in all_features:
    
        features = features
        arguments_list.append(TestParameters(dataset_name=dataset_name,
                                    features=features))

In [10]:
for d in datasets:
    print(len(d),'\t', d.name)

109539 	 chordal10
58 	 chordal6
272 	 chordal7
1614 	 chordal8
11911 	 chordal9
33120 	 eul10
16 	 eul6
54 	 eul7
243 	 eul8
2038 	 eul9
2322 	 ge10c
6 	 graph4c
34 	 graph5
156 	 graph6
1044 	 graph7


In [11]:
from tqdm import tqdm
for dataset in tqdm(datasets):
    dataset.generate_k_graph_cache(k=2)
    dataset.generate_k_graph_cache(k=3)
    dataset.generate_k_graph_cache(k=3, modified=True)


100%|██████████| 15/15 [01:33<00:00,  6.24s/it]


In [12]:
test_operator = TestOperator(single_thread=False, batch_size=24)

test_operator.tests(arguments_list)

100%|██████████| 600/600 [07:39<00:00,  1.31it/s]
